In [2]:
import requests
from bs4 import BeautifulSoup

In [3]:
import time
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.lisabeautysupply.com"
COLLECTION_URL = "https://www.lisabeautysupply.com/?s=skin+care&post_type=product"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

def get_soup(url):
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")
    except requests.HTTPError as e:
        if response.status_code == 404:
            print(f"⚠️ Page not found: {url}")
            return None
        else:
            raise  # re-raise for other errors

def extract_product_links(soup):
    if soup is None:
        return []
    product_tags = soup.find_all("a", class_="woocommerce-LoopProduct-link")
    print(f"Found {len(product_tags)} products on this page.")
    return [tag['href'] for tag in product_tags if tag.get('href')]

def scrape_all_product_links(start_url):
    page = 1
    all_links = set()

    while True:
        if page == 1:
            paged_url = start_url
        else:
            paged_url = f"{start_url}&paged={page}"

        print(f"\nScraping page {page}... {paged_url}")
        soup = get_soup(paged_url)
        links = extract_product_links(soup)

        if not links:
            print("No more products found. Stopping.")
            break

        all_links.update(links)
        page += 1
        time.sleep(2)  # be polite

    return list(all_links)

# ✅ Run the scraper
if __name__ == "__main__":
    product_links = scrape_all_product_links(COLLECTION_URL)
    print(f"\nTotal products scraped: {len(product_links)}")
    for link in product_links[:10]:  # preview first 10
        print(link)



Scraping page 1... https://www.lisabeautysupply.com/?s=skin+care&post_type=product
Found 12 products on this page.

Scraping page 2... https://www.lisabeautysupply.com/?s=skin+care&post_type=product&paged=2
Found 12 products on this page.

Scraping page 3... https://www.lisabeautysupply.com/?s=skin+care&post_type=product&paged=3
Found 12 products on this page.

Scraping page 4... https://www.lisabeautysupply.com/?s=skin+care&post_type=product&paged=4
Found 12 products on this page.

Scraping page 5... https://www.lisabeautysupply.com/?s=skin+care&post_type=product&paged=5
Found 12 products on this page.

Scraping page 6... https://www.lisabeautysupply.com/?s=skin+care&post_type=product&paged=6
Found 12 products on this page.

Scraping page 7... https://www.lisabeautysupply.com/?s=skin+care&post_type=product&paged=7
Found 8 products on this page.

Scraping page 8... https://www.lisabeautysupply.com/?s=skin+care&post_type=product&paged=8
⚠️ Page not found: https://www.lisabeautysupply.c

In [6]:
# --- Helper functions ---
def get_date_added():
    try:
        return datetime.today().strftime("%Y-%m-%d")  # e.g. 2025-08-17
    except:
        return ""


def get_product_name(product_url):
    """Extract product name (title) from product detail page"""
    soup = get_soup(product_url)
    title_tag = soup.find("h1", class_="product-title")
    if title_tag:
        return title_tag.get_text(strip=True)
    return None


import re

def get_package_size(product_name):
    """
    Extract package size (e.g., '8.4 oz', '250 ml', '3.5 oz') from product name.
    Returns list of sizes if multiple units are present.
    """
    pattern = r"(\d+(\.\d+)?\s?(oz|ml|g|kg|lb|fl|L|mL))"
    matches = re.findall(pattern, product_name, flags=re.IGNORECASE)
    return [m[0] for m in matches] if matches else None


def get_soup(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error for {url}: {e}")
    except requests.exceptions.RequestException as e:
        print(f"Request failed for {url}: {e}")
    return None


def get_sku(soup):
    sku_tag = soup.find("span", class_="sku")
    return sku_tag.get_text(strip=True) if sku_tag else None


def get_description(soup):
    desc_tag = soup.find("div", class_="woocommerce-Tabs-panel--description")
    if desc_tag:
        return desc_tag.get_text(" ", strip=True)  # join with spaces
    return None


from urllib.parse import urljoin
import random
import logging

def get_product_images(soup, base_url=None, min_size=400):
    """
    Extract all product image URLs from a BeautifulSoup object (product page).
    Uses multiple selectors and srcset handling to get higher-resolution images.
    
    Parameters:
    - soup: BeautifulSoup object of the product page
    - base_url: The page URL to resolve relative links
    - min_size: Minimum width/height to consider
    
    Returns:
    - List of unique image URLs (or empty list if none found)
    """
    urls = set()

    # Target product gallery / slider
    product_imgs = soup.select(
        "img#product-featured-image, "
        "img[id^='product-featured-image'], "
        "div.product-gallery img, "
        "div.Product__Slideshow img, "
        "div.Product__Gallery img, "
        "img.product-single__media"
    )

    # fallback: all img tags
    if not product_imgs:
        product_imgs = soup.find_all("img")

    for img in product_imgs:
        # handle srcset (different resolutions)
        if img.has_attr("srcset"):
            for candidate in img["srcset"].split(","):
                parts = candidate.strip().split(" ")
                if len(parts) == 2 and parts[1].endswith("w"):
                    try:
                        width = int(parts[1][:-1])
                        if width >= min_size:
                            src_url = parts[0]
                            if src_url.startswith("//"):
                                src_url = "https:" + src_url
                            if base_url:
                                urls.add(urljoin(base_url, src_url))
                            else:
                                urls.add(src_url)
                    except ValueError:
                        pass

        # fallback: src / data-src / data-large_image
        for attr in ["data-src", "data-large_image", "src"]:
            if img.has_attr(attr):
                src_url = img[attr]
                if src_url.startswith("//"):
                    src_url = "https:" + src_url
                full_url = urljoin(base_url, src_url) if base_url else src_url

                try:
                    w = int(img.get("width", 0))
                    h = int(img.get("height", 0))
                    if max(w, h) >= min_size:
                        urls.add(full_url)
                except Exception:
                    urls.add(full_url)

    return list(urls)



def get_product_type():
    return "skin care"


def get_source(url):
    """Return the source URL (current product page being scraped)."""
    return url


In [7]:
# --- Initialize dictionary ---
d = {col: [] for col in [
    "Product ID", "Product Name", "Product Type", "Category", "Brand Name",
    "Product Line Name", "Ingredients", "Use Instructions", "Package Size",
    "Product Description", "Product Colour", "Country of Origin", "Date Added",
    "Barcode (EAN/UPC)", "Barcode Type (e.g., EAN-13, UPC-A)", "Batch Number",
    "SKU", "Benefits", "Product Image URL", "Hero Ingriedients Match",
    "Key Ingriedients", "Key Ingriedients2", "Key Ingriedients3",
    "Key Ingriedients4", "Key Ingriedients5", "Product Images",
    "Verification Status", "Verification Date",
    "Notes (For internal use: flags, manual checks, comments, etc.)", "Source"
]}

# --- Safe helper for product name using soup ---
def get_product_name_from_soup(soup):
    if not soup:
        return None
    title_tag = soup.find("h1", class_="product-title")
    return title_tag.get_text(strip=True) if title_tag else None

# --- Loop through links ---
for idx, url in enumerate(product_links, start=1):
    print(f"🔗 Scraping [{idx}/{len(product_links)}]: {url}")

    soup = get_soup(url)
    if not soup:
        print(f"❌ Failed to load: {url}")
        # Append empty values, except Source
        for key in d:
            if key == "Source":
                d[key].append(url)
            else:
                d[key].append("")
        continue

    # --- Extract data ---
    name = get_product_name_from_soup(soup)
    package_size = get_package_size(name) if name else []
    sku = get_sku(soup)
    description = get_description(soup)
    images = get_product_images(soup)
    brand = " ".join(name.split()[:1]) if name else ""
    product_type = get_product_type()
    date_added = get_date_added()

    # --- Fill dictionary ---
    d["Product ID"].append("")
    d["Product Name"].append(name or "")
    d["Product Type"].append(product_type)
    d["Category"].append("")
    d["Brand Name"].append(brand)
    d["Product Line Name"].append("")
    d["Ingredients"].append("")
    d["Use Instructions"].append("")
    d["Package Size"].append(", ".join(package_size) if package_size else "")
    d["Product Description"].append(description or "")
    d["Product Colour"].append("")
    d["Country of Origin"].append("")
    d["Date Added"].append(date_added)
    d["Barcode (EAN/UPC)"].append("")
    d["Barcode Type (e.g., EAN-13, UPC-A)"].append("")
    d["Batch Number"].append("")
    d["SKU"].append(sku or "")
    d["Benefits"].append("")
    d["Product Image URL"].append(", ".join(images) if images else "")
    d["Hero Ingriedients Match"].append("")
    d["Key Ingriedients"].append("")
    d["Key Ingriedients2"].append("")
    d["Key Ingriedients3"].append("")
    d["Key Ingriedients4"].append("")
    d["Key Ingriedients5"].append("")
    d["Product Images"].append("")
    d["Verification Status"].append("")
    d["Verification Date"].append("")
    d["Notes (For internal use: flags, manual checks, comments, etc.)"].append("")
    d["Source"].append(url)

# --- Convert to DataFrame ---
import pandas as pd
df = pd.DataFrame(d)
print("✅ DataFrame created:", df.shape)

🔗 Scraping [1/80]: https://www.lisabeautysupply.com/product/skin-light-body-lotion-with-carrot-extract-and-vitamin-e-16-9-oz/
🔗 Scraping [2/80]: https://www.lisabeautysupply.com/product/aveeno-positively-radiant-skin-brightening-exfoliating-face-scrub-7-oz/
🔗 Scraping [3/80]: https://www.lisabeautysupply.com/product/pr-francoise-bedon-supreme-lightening-beauty-cream-with-argan-oil-1-69-oz/
🔗 Scraping [4/80]: https://www.lisabeautysupply.com/product/nutriclair-carrot-clarifying-moisturizing-milk-17-oz/
🔗 Scraping [5/80]: https://www.lisabeautysupply.com/product/easy-white-express-carrot-radiance-and-clarity-body-lotion-16-9-oz/
🔗 Scraping [6/80]: https://www.lisabeautysupply.com/product/clear-quick-active-gel-3-45-oz/
🔗 Scraping [7/80]: https://www.lisabeautysupply.com/product/pure-skin-black-spot-corrector-lotion-1-oz/
🔗 Scraping [8/80]: https://www.lisabeautysupply.com/product/clean-clear-morning-burst-skin-brightening-facial-scrub-oil-free-5-oz/
🔗 Scraping [9/80]: https://www.lisabea

In [8]:
df.to_excel('lisabeauty.xlsx', index=False)

In [17]:
url = "https://www.lisabeautysupply.com/product/g-g-dynamiclair-beauty-soap-6-7-oz-200g/"
soup = get_soup(url)

images = get_product_images(soup)
print(", ".join(images) if images else "No images found")


https://www.lisabeautysupply.com/wp-content/uploads/2023/03/yhst-88462588038071_2613_1660495060-100x100.png, https://www.lisabeautysupply.com/wp-content/uploads/2023/03/yhst-88462588038071_2613_2278082030-247x296.png, https://www.lisabeautysupply.com/wp-content/uploads/2023/03/yhst-88462588038071_2615_11055502736-247x296.png, https://www.lisabeautysupply.com/wp-content/uploads/2018/12/lisa-logo2-2.png, https://www.lisabeautysupply.com/wp-content/uploads/2023/03/yhst-88462588038071_2612_9865143318-196x296.png, https://www.lisabeautysupply.com/wp-content/uploads/2023/03/yhst-88462588038071_2613_1660264574-100x100.png, https://www.lisabeautysupply.com/wp-content/uploads/2023/03/yhst-88462588038071_2612_923449960-247x254.png, https://www.lisabeautysupply.com/wp-content/uploads/2023/03/yhst-88462588038071_2613_1660364430.jpg, https://www.lisabeautysupply.com/wp-content/uploads/2023/03/yhst-88462588038071_2615_11059016581-247x296.png, https://www.lisabeautysupply.com/wp-content/uploads/2023/